Model

In [1]:
import anndata
import numpy as np
import pandas as pd
from scipy.sparse import spmatrix
import scanpy as sc
from sklearn.preprocessing import StandardScaler
from celltypist import models

import logging
logger = logging.getLogger("Data-Preprocessing")

Data Preprocessing

In [2]:
data_path = "./data/global.h5ad"
raw = anndata.read_h5ad(data_path)
logger.info("🍳 Preparing data before training")

old_to_new_class_map = {}
with open("/home/lagandua/projects/sven/pineapple/cell_mapping2.txt", "r") as file:
    for line in file:
        line = line.strip()
        split_line = line.split(":")
        old_classes = split_line[1].split(",")
        new_class = split_line[0]
        for c in old_classes:
            old_to_new_class_map[c] = new_class

🍳 Preparing data before training


In [3]:

# Create new class labels
# label = "sven_classes"
# raw.obs[label] = [
#     old_to_new_class_map[c] for c in raw.obs["Predicted_labels_CellTypist"]
# ]

label = "Predicted_labels_CellTypist"

In [4]:
# Value counts of new class labels
logger.info(f"🍳 Total classes {len(raw.obs[label].unique())}")
logger.info("🍳 Value counts of new class labels"
            f"\n{raw.obs[label].value_counts()}")

🍳 Total classes 91
🍳 Value counts of new class labels
Predicted_labels_CellTypist
Cytotoxic T cells    36883
Memory B cells       31194
Helper T cells       25761
Macrophages          24071
CD16+ NK cells       21495
                     ...  
Transitional DC         41
Cycling monocytes       31
Myelocytes              25
Pre-B cells             21
Cycling DCs             18
Name: count, Length: 91, dtype: int64


In [5]:

# Only keep the classes that have 2000 or more cells
# min_cells_per_class = 2000
classes = raw.obs[label].value_counts().index
# classes = classes[classes >= min_cells_per_class].index
# logger.info(f"🍳 {len(set(classes))} Classes with {min_cells_per_class} or more cells"
#             f"\n{classes}")

In [6]:
# Create a gene filter with scanpy's highly_variable_genes
n_top_genes = 10000
gene_filer = sc.pp.highly_variable_genes(raw, flavor="seurat_v3", n_top_genes=n_top_genes, inplace=False)

In [7]:
indata = raw.X[raw.obs[label].isin(classes)]
genes = raw.var_names
labels = raw.obs[raw.obs[label].isin(classes)][label]

if isinstance(indata, pd.DataFrame):
    indata = indata.values
elif isinstance(indata, spmatrix):
    indata = indata.toarray()

labels = np.array(labels)
genes = np.array(genes)

False

In [8]:
# Filter genes based on the celltypist model
# models.models_path = "/home/lagandua/projects/sven/pineapple/celltypist/.cache"
# ct_model = models.Model.load(model = 'Immune_All_Low.pkl')
# chosen_genes = np.array(
#         list(set(ct_model.classifier.features).intersection(set(raw.var_names)))
#     )
# keep_model = np.isin(raw.var_names, chosen_genes)

# Filter genes based on the highly variable genes
keep_model = gene_filer["highly_variable"].values
flag = indata.sum(axis = 0) == 0
if flag.sum() > 0:
    logger.info(f"✂️ {flag.sum()} non-expressed are filtered out")
    keep = (~flag) & keep_model
    genes = genes[keep]
logger.info(f"🔬 Input data has {indata.shape[0]} cells and {(keep).sum()} genes")

✂️ 3912 non-expressed are filtered out
🔬 Input data has 329762 cells and 10000 genes


In [19]:
# Scaler (this takes forever to run)
# Set to False if you are want to use the data for VAE with ZINB loss
# Set to True if you are want to use the data for VAE with MSE loss
scale_data = True
if scale_data:
    logger.info(f"⚖️ Scaling input data")
    scaler = StandardScaler(with_mean = True)
    indata = scaler.fit_transform(indata[:, keep] if keep.sum() > 0 else indata)
    indata[indata > 10] = 10
else:
    logger.info(f"⚖️ Not scaling the input data")
    indata = indata[:, keep] if keep.sum() > 0 else indata

⚖️ Scaling input data


In [20]:
# Change the data name based on the what dataset you are saving
data_name = f"low_top_{n_top_genes}_genes"
# data_name = f"high_all_genes_sparse"

# Save preprocessed data into train and test sets (this also takes forever)
logger.info(f"📝 Saving preprocessed data into full, train and validation sets")

val_size = 0.1 # 10%
true_count = int(indata.shape[0] * val_size)
false_count = indata.shape[0] - true_count

val_flag = np.concatenate((np.full(true_count, True), np.full(false_count, False)))
np.random.shuffle(val_flag)

np.savez(f"./data/{label}/data_{data_name}.npz", data = indata, labels = labels, genes = genes)
np.savez(f'./data/{label}/train_{data_name}_{int(100 * (1-val_size))}.npz', data=indata[~val_flag], labels=labels[~val_flag], genes=genes)
np.savez(f'./data/{label}/val_{data_name}_{int(100 * (val_size))}.npz', data=indata[val_flag], labels=labels[val_flag], genes=genes)
logger.info(f"DONE! 🎉")

📝 Saving preprocessed data into full, train and validation sets
DONE! 🎉
